# OmniRail Engine — Uji Tahap 0 di Google Colab

Notebook ini mengulang pengujian Tahap 0 dari repo [OmniRail-Engine](https://github.com/devaldicaliesta/OmniRail-Engine), tanpa kunci API dan tanpa biaya token:

1. Pemeriksaan dataset uji dan test unit
2. Mencoba prototipe detektor dengan teks Anda sendiri
3. Evaluasi prototipe OmniRail lawan Presidio pada split dev, dengan interval Wilson 95% dan uji McNemar
4. Grafik perbandingan dan contoh kasus gagal
5. *(Opsional)* Uji LiteLLM + guardrail Presidio sebagai proxy, terhadap server tiruan
6. *(Opsional)* Menghitung hasil tinjauan label T-007

Jalankan sel dari atas ke bawah (**Runtime → Run all**). Semua data di dataset adalah **data rekaan**. Holdout tidak dibaca.

## 0. Persiapan

In [ ]:
CABANG = "main"  # @param {type:"string"}

!git clone -q --depth 1 --branch {CABANG} https://github.com/devaldicaliesta/OmniRail-Engine.git 2>/dev/null || echo "Repo sudah ada"
%cd OmniRail-Engine
!git log --oneline -1
!pip install -q presidio-analyzer==2.2.366

In [ ]:
import sys

sys.path.insert(0, "eval/tools")

import pandas as pd
from IPython.display import Markdown, display

import evaluasi as e
import prototipe_detektor as p

## 1. Dataset uji dan test unit

Pemeriksa memastikan format, label, validator struktur (Luhn, NIK, KK, NIP, dan lainnya), tidak ada kebocoran dev ke holdout, dan tabel jumlah di `eval/README.md` cocok. Lima peringatan telepon kantor `(021) …` memang disengaja (di luar cakupan MVP).

In [ ]:
!python eval/tools/cek_dataset.py --ringkasan

In [ ]:
!python -m unittest discover -s eval/tools

## 2. Coba prototipe detektor

Ubah isi `TEKS` dan, bila perlu, `PESAN_SEBELUMNYA` (untuk menguji kata konteks lintas pesan), lalu jalankan selnya. Prototipe ini adalah acuan Python, bukan kode produk.

In [ ]:
def periksa(teks, pesan_sebelumnya=None):
    temuan = p.deteksi_pesan(teks, pesan_sebelumnya or None)
    if not temuan:
        return pd.DataFrame({"hasil": ["tidak ada temuan"]})
    return pd.DataFrame([{"jenis": t.jenis, "nilai": t.nilai, "posisi": f"{t.mulai}-{t.akhir}"} for t in temuan])


TEKS = "NIK saya 5171041203944539, HP 0812-3456-7801, email budi@example.com"  # @param {type:"string"}
PESAN_SEBELUMNYA = ""  # @param {type:"string"}

periksa(TEKS, PESAN_SEBELUMNYA)

In [ ]:
contoh = [
    ("Pembayaran dengan kartu 5171041203944539 berhasil", None),
    ("No. KK 7250270205242406, NIK kepala keluarga 5245280311400260", None),
    ("Mohon maaf lahir dan batin, 10-04-2026", None),
    ("Nomor pesanan 2311093234567890 siap dikirim", None),
    ("1234567890", "Boleh minta nomor rekeningnya untuk refund?"),
    ("1234567890", "Boleh minta nomor pesanannya?"),
]
for teks, lalu in contoh:
    hasil = [(t.jenis, t.nilai) for t in p.deteksi_pesan(teks, lalu)]
    print(f"{teks!r:<70} -> {hasil or 'tidak ada temuan'}")

## 3. Evaluasi pada split dev

Metrik dihitung sesuai aturan `eval/README.md`. Angka dalam kurung adalah interval Wilson 95%.

**Catatan penting:** angka prototipe optimistis, karena dataset dan prototipe disusun dari spesifikasi yang sama dan prototipe disetel di dev. Baseline Presidio sengaja tanpa validator. Lihat `eval/results/README.md` untuk batas kesimpulannya.

In [ ]:
hasil_prototipe = e.jalan("prototipe")
display(Markdown(e.markdown("prototipe", hasil_prototipe)))

In [ ]:
hasil_presidio = e.jalan("presidio")
display(Markdown(e.markdown("presidio", hasil_presidio)))

In [ ]:
display(Markdown(e.banding("prototipe", "presidio")))

## 4. Grafik perbandingan

Titik adalah nilai, garis adalah interval Wilson 95%. Baris teratas adalah gabungan lima entitas kriteria T-003 (`nik`, `kk`, `credit_card`, `bpjs`, `bank_account`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PERMUKAAN, TEKS_UTAMA, TEKS_KEDUA, GARIS = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SISTEM = [
    ("prototipe", "Prototipe OmniRail", "#2a78d6", "o", hasil_prototipe),
    ("presidio", "Presidio + regex sederhana", "#eb6834", "s", hasil_presidio),
]
UKURAN = [("recall", "Recall"), ("precision", "Precision"), ("false_positive", "False positive (baris negatif)")]
BARIS = ["gabungan T-003"] + list(hasil_prototipe["per_entitas"])


def angka(hasil, baris, ukuran):
    sumber = hasil["gabungan"]["t003"] if baris == "gabungan T-003" else hasil["per_entitas"][baris]
    return sumber.get(ukuran)


plt.rcParams.update({"font.size": 10, "text.color": TEKS_UTAMA, "axes.labelcolor": TEKS_KEDUA})
fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharey=True, facecolor=PERMUKAAN)
for ax, (kunci, judul) in zip(axes, UKURAN):
    ax.set_facecolor(PERMUKAAN)
    for i, (_, nama, warna, penanda, hasil) in enumerate(SISTEM):
        geser = -0.18 if i == 0 else 0.18
        for j, b in enumerate(BARIS):
            a = angka(hasil, b, kunci)
            if not a:
                continue
            ax.plot(a["ci95"], [j + geser] * 2, color=warna, lw=2, solid_capstyle="round", zorder=2)
            ax.plot(a["nilai"], j + geser, penanda, color=warna, ms=8, mec=PERMUKAAN, mew=2, zorder=3,
                    label=nama if j == 0 else None)
            if j == 0:
                ax.annotate(f"{a['nilai']:.3f}", (a["ci95"][1], j + geser), xytext=(6, 0),
                            textcoords="offset points", va="center", color=TEKS_KEDUA, fontsize=9)
    ax.axhline(0.5, color=GARIS, lw=1)
    ax.set_title(judul, loc="left", fontsize=12, color=TEKS_UTAMA)
    ax.set_xlim(0, 1.08)
    ax.grid(axis="x", color=GARIS, lw=1)
    ax.set_axisbelow(True)
    for sisi in ("top", "right", "left"):
        ax.spines[sisi].set_visible(False)
    ax.spines["bottom"].set_color(GARIS)
    ax.tick_params(colors=TEKS_KEDUA, length=0)
axes[0].set_yticks(np.arange(len(BARIS)), BARIS)
axes[0].invert_yaxis()
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper left", ncol=2, frameon=False, bbox_to_anchor=(0.01, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 5. Contoh kasus gagal

Kasus gagal tiap sistem, beserta teks aslinya dari split dev. `terlewat` berarti entitas tidak terdeteksi; `salah tandai` berarti ada temuan yang tidak diharapkan atau jenisnya keliru.

In [ ]:
import csv

dev = {r["id"]: r for r in csv.DictReader(open("eval/dev/uji_pii_id.csv", encoding="utf-8"))}


def kasus_gagal(hasil, n=15):
    df = pd.DataFrame(hasil["kasus_gagal"])
    if df.empty:
        return df
    df["teks"] = df["id"].map(lambda i: dev[i]["teks"][:110].replace("\n", " / "))
    df["diharapkan"] = df["id"].map(lambda i: dev[i]["diharapkan"][:60])
    return df


gagal_presidio = kasus_gagal(hasil_presidio)
display(gagal_presidio.groupby(["jenis", "entitas"]).size().sort_values(ascending=False).rename("jumlah").to_frame())
display(gagal_presidio.head(15))

In [ ]:
kasus_gagal(hasil_prototipe)

## 6. *(Opsional)* Uji LiteLLM + Presidio sebagai proxy (T-005)

Menjalankan LiteLLM proxy sungguhan dengan guardrail Presidio terhadap server tiruan OpenAI dan Anthropic: pemulihan token, streaming, token buatan pengguna, credentials, dan overhead latensi. Butuh sekitar 5 menit dan memasang paket tambahan. Centang `JALANKAN_LITELLM` untuk menjalankannya.

In [ ]:
JALANKAN_LITELLM = False  # @param {type:"boolean"}

if JALANKAN_LITELLM:
    !pip install -q "litellm[proxy]==1.109.0" presidio-anonymizer==2.2.366 flask
    !python eval/tools/uji_litellm.py > /dev/null 2>&1
    import json

    t005 = json.load(open("eval/results/t005-litellm.json", encoding="utf-8"))
    baris = []
    for konfigurasi in ("default", "presidio_filter_scope_input"):
        for s in t005[konfigurasi]["skenario"]:
            baris.append({"konfigurasi": konfigurasi, "skenario": s["skenario"], "status": s["status"],
                          "nilai asli sampai provider": s["nilai_asli_sampai_provider"],
                          "dipulihkan": s.get("nilai_dipulihkan"), "diblokir": s.get("diblokir")})
    display(pd.DataFrame(baris))
    display(pd.DataFrame({k: t005[k]["latensi"] for k in ("presidio_filter_scope_input", "tanpa_guardrail")}).T)
else:
    print("Dilewati. Centang JALANKAN_LITELLM lalu jalankan sel ini.")

## 7. *(Opsional)* Hitung hasil tinjauan label (T-007)

Isi salinan `eval/tinjauan/t007-formulir.csv` mengikuti `eval/tinjauan/README.md`, beri nama `t007-formulir-NAMA.csv`, lalu unggah lewat sel ini.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

if files is None:
    print("Sel ini hanya untuk Google Colab.")
else:
    for nama in files.upload():
        !python eval/tools/tinjau_label.py nilai "{nama}"